# Create golf-bunker dataset from OpenStreetMap data and Mapbox tiles

Selects the top-N best-mapped golf courses (most `golf=bunker` ways tagged inside the course polygon) from a region, then downloads the bunker geometries for those courses plus matching Mapbox satellite tiles.

Picking a small number of well-mapped courses is deliberate: OSM bunker completeness varies wildly course-to-course, and untagged-but-visible bunkers in training tiles teach the model to *suppress* bunker-shaped detections. Restricting to courses with high tagged-bunker counts is a cheap proxy for completeness.

**This notebook is CPU/network-only — no GPU needed.** Runs equally well locally or on Colab. Locally is faster and survives across sessions; Colab is more turnkey if you don't have a Python environment set up.

**Local setup:**
```bash
uv pip install -e .   # from the repo root, once
export MAPBOX_TOKEN=...
export HF_TOKEN=...
jupyter lab demo/create_bunker_dataset.ipynb
```

## Dependencies

In [ ]:
import sys
if "google.colab" in sys.modules:
    %pip install --quiet git+https://github.com/dkozickis/osm-ai-helper.git@bunker-detector
    # On Colab the package is fresh per runtime; locally install it once via
    # `uv pip install -e .` (or `pip install -e .`) from the repo root and skip this cell.

## Find best-mapped golf courses and download bunker geometries

In [ ]:
from osm_ai_helper.find_best_mapped_courses import find_best_mapped_courses

- `REGION`

OSM area name used as the search region. UK has the densest mapping of well-tagged courses; Ireland and the Netherlands are also good.

- `N_TRAIN` / `N_VAL`

Number of top-ranked courses (by tagged bunker count) to use for each split. The next-ranked `N_VAL` courses are used for validation. They are geographically separate from the train set by construction (different courses).

- `SELECTOR` / `CLASS_NAME`

OSM tag and human-readable class name. Used downstream by the YOLO config and HF dataset README.

In [ ]:
REGION = "United Kingdom"
N_TRAIN = 8
N_VAL = 2
SELECTOR = "golf=bunker"
CLASS_NAME = "bunker"
TRAIN_DIR = "courses_train"
VAL_DIR = "courses_val"

In [ ]:
find_best_mapped_courses(
    output_dir="datasets",
    region=REGION,
    n_train=N_TRAIN,
    n_val=N_VAL,
    # Pettens Links sits in a sand-dune zone where natural bunkers are
    # huge — out of distribution for the model trained on smaller pot-
    # bunker style courses. Replaced by the next-ranked course (Wentworth).
    skip_course_ids=[204317950],
)

Inspect the per-course summary so you know which courses are in train vs val and how many bunkers each contributes.

In [ ]:
import json

with open("datasets/courses_summary.json") as f:
    summary = json.load(f)
for course in summary:
    print(f"#{course['rank']:>2} [{course['split']}] {course['name'] or '(unnamed)'} — {course['bunker_count']} bunkers")

## Download tiles from Mapbox

In [ ]:
import os
import sys

from osm_ai_helper.group_elements_and_download_tiles import (
    group_elements_and_download_tiles,
)

if "google.colab" in sys.modules:
    from google.colab import userdata
    MAPBOX_TOKEN = MAPBOX_TOKEN
    HF_TOKEN = HF_TOKEN
else:
    MAPBOX_TOKEN = os.environ["MAPBOX_TOKEN"]
    HF_TOKEN = os.environ["HF_TOKEN"]

- `ZOOM`

Bunkers are smaller and lower-contrast than swimming pools, so we use zoom 19 (vs 18 for pools). Higher zoom = each tile covers a smaller area but bunkers are easier to see. At z19 over 10 courses the tile count is well within the Mapbox free tier.

In [ ]:
ZOOM = 19

Set the `MAPBOX_TOKEN`:

- **Locally:** `export MAPBOX_TOKEN=...` before launching Jupyter.
- **On Colab:** set as a notebook secret via the 🔑 sidebar (with notebook access on).

Get your [Default Public Token](https://docs.mapbox.com/help/getting-started/access-tokens/#your-default-public-token) from https://console.mapbox.com/.

Mapbox satellite imagery is licensed for tracing into OSM ([reference](https://docs.mapbox.com/data/tilesets/guides/imagery/#trace-satellite-imagery)).

In [ ]:
group_elements_and_download_tiles(
    "datasets/courses_train.json",
    f"datasets/{TRAIN_DIR}",
    MAPBOX_TOKEN,
    zoom=ZOOM,
)

In [ ]:
group_elements_and_download_tiles(
    "datasets/courses_val.json",
    f"datasets/{VAL_DIR}",
    MAPBOX_TOKEN,
    zoom=ZOOM,
)

## Convert to YOLO dataset

Convert tile + element JSON pairs into the [YOLO format](https://docs.ultralytics.com/datasets/detect/) (`.txt` label files alongside each `.jpg`).

In [ ]:
from osm_ai_helper.convert_to_yolo_dataset import convert_to_yolo_dataset

In [ ]:
convert_to_yolo_dataset(f"datasets/{TRAIN_DIR}")

In [ ]:
convert_to_yolo_dataset(f"datasets/{VAL_DIR}")

# Spot-check annotations

Plot a tile with its YOLO bounding boxes overlayed. Bunkers should land on actual sand. If boxes are on greens/fairways or appear shifted, something is wrong with the coordinate conversion.

In [ ]:
import random
from pathlib import Path

from PIL import Image, ImageDraw

TILE_SIZE = 512

def show_with_boxes(img_path):
    img = Image.open(img_path).convert('RGB')
    label_path = img_path.with_suffix('.txt')
    draw = ImageDraw.Draw(img)
    for line in label_path.read_text().strip().splitlines():
        _, cx, cy, w, h = map(float, line.split())
        x1 = (cx - w / 2) * TILE_SIZE
        y1 = (cy - h / 2) * TILE_SIZE
        x2 = (cx + w / 2) * TILE_SIZE
        y2 = (cy + h / 2) * TILE_SIZE
        draw.rectangle([x1, y1, x2, y2], outline='red', width=2)
    return img

tile_paths = list(Path(f'datasets/{TRAIN_DIR}').glob('*.jpg'))
show_with_boxes(random.choice(tile_paths))

# Check out-of-the-box predictions (sanity check)

Stock YOLO11 doesn't know what a bunker is — these predictions should look bad / random. That's expected; finetuning happens in the next notebook.

In [ ]:
from pathlib import Path
from ultralytics import YOLO

In [ ]:
yolo = YOLO("yolo11m.pt")

In [ ]:
yolo.predict(list(Path(f"datasets/{VAL_DIR}").glob("*.jpg"))[0], save=True)

In [ ]:
from PIL import Image

Image.open(list(Path("runs/detect/predict").glob("*.jpg"))[0])

# Upload Dataset

The dataset will be uploaded to [HuggingFace Datasets](https://huggingface.co/docs/hub/datasets).

Set the `HF_TOKEN`:

- **Locally:** `export HF_TOKEN=...` before launching Jupyter.
- **On Colab:** set as a notebook secret via the 🔑 sidebar.

Get a write-scoped token from https://huggingface.co/settings/tokens.

In [ ]:
!rm -f "datasets/{TRAIN_DIR}"/*.json

In [ ]:
!rm -f "datasets/{VAL_DIR}"/*.json

In [ ]:
!zip -r -q train.zip "datasets/{TRAIN_DIR}"

In [ ]:
!zip -r -q val.zip "datasets/{VAL_DIR}"

Set `USER` to your HuggingFace username. The dataset repo `{USER}/osm-golf-bunkers` will be created if it doesn't exist.

In [ ]:
USER = "JohnieWalkerLV"
REPO = "osm-golf-bunkers"

Create the YAML config consumed by YOLO.

In [ ]:
Path("yolo_dataset.yaml").write_text(
    f"""
path: .
train: {TRAIN_DIR}
val: {VAL_DIR}

names:
  0: {CLASS_NAME}
"""
)

In [ ]:
Path("README.md").write_text(
    f"""
---
task_categories:
- object-detection
---

# {REPO}

Detect {CLASS_NAME}s (sand traps) on golf courses in satellite imagery.

Created with [osm-ai-helper](https://github.com/mozilla-ai/osm-ai-helper).

## Source courses

Top {N_TRAIN + N_VAL} best-mapped golf courses in {REGION} (ranked by tagged `golf=bunker` count inside `leisure=golf_course` polygons).
First {N_TRAIN} courses are used for training; the next {N_VAL} for validation.

## Ground Truth Bounding Boxes

Downloaded from [OpenStreetMap](https://www.openstreetmap.org). LICENSE: https://www.openstreetmap.org/copyright

Used the `{SELECTOR}` [OpenStreetMap tag](https://wiki.openstreetmap.org/wiki/Map_features).

## Satellite Images

Downloaded from [Mapbox](https://www.mapbox.com/). LICENSE: https://docs.mapbox.com/data/tilesets/guides/imagery/#trace-satellite-imagery

Used a [zoom level](https://docs.mapbox.com/help/glossary/zoom-level/) of `{ZOOM}`.
"""
)

In [ ]:
from huggingface_hub import HfApi

In [ ]:
api = HfApi()

In [ ]:
try:
    api.create_repo(f"{USER}/{REPO}", token=HF_TOKEN, repo_type="dataset")
except Exception:
    pass

In [ ]:
api.upload_file(
    token=HF_TOKEN,
    path_or_fileobj="train.zip",
    path_in_repo="train.zip",
    repo_id=f"{USER}/{REPO}",
    repo_type="dataset",
)

In [ ]:
api.upload_file(
    token=HF_TOKEN,
    path_or_fileobj="val.zip",
    path_in_repo="val.zip",
    repo_id=f"{USER}/{REPO}",
    repo_type="dataset",
)

In [ ]:
api.upload_file(
    token=HF_TOKEN,
    path_or_fileobj="yolo_dataset.yaml",
    path_in_repo="yolo_dataset.yaml",
    repo_id=f"{USER}/{REPO}",
    repo_type="dataset",
)

In [ ]:
api.upload_file(
    token=HF_TOKEN,
    path_or_fileobj="README.md",
    path_in_repo="README.md",
    repo_id=f"{USER}/{REPO}",
    repo_type="dataset",
)

In [ ]:
api.upload_file(
    token=HF_TOKEN,
    path_or_fileobj="datasets/courses_summary.json",
    path_in_repo="courses_summary.json",
    repo_id=f"{USER}/{REPO}",
    repo_type="dataset",
)